[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hany73-max/PDE-Rover-Engine/blob/main/labs/Lab_3_Nonlinear_Charpit.ipynb) [![Launch JupyterLab](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/hany73-max/PDE-Rover-Engine/main?urlpath=lab/tree/labs/Lab_3_Nonlinear_Charpit.ipynb)


In [ ]:
# 🚀 ENVIRONMENT & PATH SETUP (Works in local JupyterLab, MyBinder, VS Code, and Google Colab)
import os, sys
try:
    if 'google.colab' in str(get_ipython()):
        if not os.path.exists('PDE-Rover-Engine'):
            import subprocess
            subprocess.run(['git', 'clone', 'https://github.com/hany73-max/PDE-Rover-Engine.git'], check=True)
        else:
            import subprocess
            subprocess.run(['git', '-C', 'PDE-Rover-Engine', 'pull', '--ff-only'], check=False)
        os.chdir('PDE-Rover-Engine')
except NameError:
    pass

# Automatically locate the repository root (containing labs/ and rover_path_finder/)
# regardless of whether JupyterLab sets cwd to the repo root, labs/, or solutions/labs/:
_cwd = os.path.abspath(os.getcwd())
for _candidate in [_cwd, os.path.abspath(os.path.join(_cwd, '..')), os.path.abspath(os.path.join(_cwd, '..', '..'))]:
    if os.path.isdir(os.path.join(_candidate, 'labs', 'tests')) and os.path.isdir(os.path.join(_candidate, 'rover_path_finder')):
        if _candidate not in sys.path:
            sys.path.insert(0, _candidate)
        break
print('[OK] Project root added to sys.path!')


# 📻 Lab 3: Non-Linear First-Order PDEs, Charpit's Method, and Obstacle Safety Buffers
### *Mission Control: Euclidean Distance Transforms & The Eikonal Equation*
**Course:** Applied Partial Differential Equations & Planetary Robotics
**Lecture Notes Reference:** Chapter 1, Sections 1.3.2–1.3.6 — **Notes pp. 16–26** *(PDF pp. 17–27)*

---
### 🗺️ Where You Are in the Rover Navigation Pipeline
```
[Lab 1: Terrain Slopes] ──► [Lab 2: Streamline Paths] ──► 📍 [Lab 3: Eikonal Distance & Safety Buffers] ──► [Lab 4–5: Laplace Theory] ──► [Lab 6: Complete Rover Planner]
```

---
### 🧭 The Rosetta Stone: Connecting Lecture Notes to the Rover Engine

| 1. Lecture Notes Math (PDF pp. 17–27) | 2. Geometric & Physical Meaning | 3. Real Project Function (`rover_path_finder/`) |
| :--- | :--- | :--- |
| $f(x, y, z, p, q) = 0$ (non-linear) | 1st-order PDE containing powers or products like $p^2, q^2, pq$ | Euclidean distance & wavefront propagation |
| $p^2 + q^2 = 1$ (or $\|\nabla z\| = 1$) | **Eikonal Equation:** a distance function that grows at unit rate ($1\text{ m/m}$) away from a source | [`terrain.dilate(raw, rover_radius)`](rover_path_finder/terrain.py) (expanding obstacles by the rover's body radius) |
| $z = p x + q y + f(p, q)$ | **Clairaut's Form:** complete integral is $z = a x + b y + f(a, b)$ | Family of tangent plane wavefronts |
| $(x - x_0)^2 + (y - y_0)^2 \le R^2$ | Circular safety buffer of radius $R$ around every rock/crater pixel | `yy**2 + xx**2 <= radius**2` in `terrain.dilate` |

---
## 1. Intuition First: Why Does the Rover Need the Eikonal Equation ($p^2 + q^2 = 1$)?

A Mars rover is not a zero-size point—it has a physical width (`rover_radius`). If our path planner routes the rover's center right along the edge of a crater, its wheels will clip the crater rim!

Before solving for the navigation potential in [`terrain.py`](rover_path_finder/terrain.py), the function [`dilate(raw, radius)`](rover_path_finder/terrain.py) expands every hazard pixel outward by the **Euclidean distance** $R = \text{rover\_radius}$:
* If $z(x, y) = \sqrt{x^2 + y^2}$ denotes the Euclidean distance from a point obstacle at the origin, its partial derivatives are:
  $$p = \frac{\partial z}{\partial x} = \frac{x}{\sqrt{x^2 + y^2}}, \qquad q = \frac{\partial z}{\partial y} = \frac{y}{\sqrt{x^2 + y^2}}$$
* Squaring and adding them yields the famous **Eikonal Equation**:
  $$p^2 + q^2 = 1 \quad \Longleftrightarrow \quad \|\nabla z\| = 1$$

Because $p$ and $q$ are **squared**, this 1st-order PDE is **non-linear**, so Lagrange's method from Lab 2 cannot solve it! Instead, we use the **Four Standard Non-Linear Forms** and **Charpit's General Method** (Notes pp. 16–26 / PDF pp. 17–27).


In [ ]:
import os, sys
for _p in ['.', '..', '../..']:
    _ap = os.path.abspath(_p)
    if os.path.isdir(os.path.join(_ap, 'labs', 'tests')) and _ap not in sys.path:
        sys.path.insert(0, _ap)
import numpy as np
import matplotlib.pyplot as plt
from labs.tests import test_lab3

%matplotlib inline
print("Lab 3 Environment Initialized! Ready for Non-Linear PDEs & Safety Buffers.")


---
## 2. The Mathematical Recipe: All 4 Standard Forms & Charpit's General Method (PDF pp. 17–27)

For a non-linear 1st-order PDE $f(x, y, z, p, q) = 0$, our goal is to find a **Complete Integral**—a solution containing **two arbitrary constants $(a, b)$** matching the number of independent variables $(x, y)$.

---
### 🧭 Quick Decision Table: Which Non-Linear Method Should You Use?

Before jumping into long algebra, inspect **which variables appear in the PDE**:

| Method in Notes (PDF pp. 17–27) | How to Recognize the PDE Immediately | What Appears in the PDE? | Fast Solution Recipe |
| :--- | :--- | :--- | :--- |
| **Form 1** *(Notes p. 16 / PDF p. 17)* | $f(p, q) = 0$ | **Only $p$ and $q$** (no $x, y, z$) | Set $p = a$, solve $f(a, q) = 0 \implies q = \phi(a)$, write $z = a x + \phi(a) y + c$ |
| **Form 2 (Clairaut)** *(Notes p. 18 / PDF p. 19)* | $z = p x + q y + f(p, q)$ | $z - px - qy$ equals a function of $(p, q)$ | Replace $p \to a$ and $q \to b$ directly: $z = a x + b y + f(a, b)$ *(no $+c$!)* |
| **Form 3** *(Notes p. 20 / PDF p. 21)* | $f(z, p, q) = 0$ | **Only $z, p, q$** ($x$ and $y$ are missing!) | Combine $u = x + a y \implies p = \frac{dz}{du},\; q = a\frac{dz}{du}$ and integrate the ODE for $z(u)$ |
| **Form 4 (Separable)** *(Notes p. 22 / PDF p. 23)* | $f_1(x, p) = f_2(y, q)$ | $z$ is missing, and $(x, p)$ separates from $(y, q)$ | Set $f_1(x, p) = f_2(y, q) = a$, solve for $p(x,a)$ and $q(y,a)$, integrate $dz = p\,dx + q\,dy$ |
| **Charpit's General Method** *(Notes p. 24 / PDF p. 25)* | $f(x, y, z, p, q) = 0$ | General coupling (all 5 variables or mixed $x,y,z$) | Integrate Charpit's 5-fraction auxiliary system to find $q = \psi(x,y,z,p,a)$, then integrate $dz = p\,dx + q\,dy$ |

---
### Part A: Step-by-Step Walkthrough of Each Standard Form (PDF pp. 17–25)

#### 🔹 Form 1: Equations Involving Only $p$ and $q$ — $f(p, q) = 0$ (Notes pp. 16–18 / PDF pp. 17–19)
* **Why it works:** If $p = a$ and $q = b$ are constants satisfying $f(a, b) = 0 \implies b = \phi(a)$, then integrating $dz = p\,dx + q\,dy = a\,dx + \phi(a)\,dy$ immediately gives the plane family:
  $$\boxed{z = a x + \phi(a) y + c}$$
* **📖 Worked Example 1 (Notes p. 17 / PDF p. 18):** Solve $p + q = p q$.
  1. Only $p$ and $q$ appear! Set $p = a$ and $q = b$, where $a + b = a b$.
  2. Solve for $b$ in terms of $a$: $a = b(a - 1) \implies b = \frac{a}{a - 1}$.
  3. Substitute into $z = ax + by + c$: $\boxed{z = a x + \frac{a}{a - 1}y + c}$.
* **📖 Worked Example 2 (Notes p. 17 / PDF p. 18):** Solve $p q = k$ (where $k$ is a given constant).
  1. Set $p = a, q = b$ with $a b = k \implies b = \frac{k}{a}$.
  2. Complete integral: $\boxed{z = a x + \frac{k}{a}y + c}$.

---
#### 🔹 Form 2: Clairaut's Equation — $z = p x + q y + f(p, q)$ (Notes pp. 18–19 / PDF pp. 19–20)
* **Rule:** Whenever the PDE can be written with $z = p x + q y + f(p, q)$, simply replace $p \to a$ and $q \to b$:
  $$\boxed{z = a x + b y + f(a, b)}$$
* **📖 Worked Example 2 (Notes p. 19 / PDF p. 20 — Rearranging into Clairaut's Form!):** Solve $(p + q)(z - x p - y q) = 1$.
  1. At first glance this doesn't look like Clairaut's form! Divide both sides by $(p + q)$ and isolate $z$:
     $$z - x p - y q = \frac{1}{p + q} \implies z = p x + q y + \frac{1}{p + q}$$
  2. Now it is in exact Clairaut form with $f(p, q) = \frac{1}{p + q}$! Replace $p \to a, q \to b$:
     $$\boxed{z = a x + b y + \frac{1}{a + b}}$$

---
#### 🔹 Form 3: Equations Not Involving $x$ and $y$ — $f(z, p, q) = 0$ (Notes pp. 20–22 / PDF pp. 21–23)
* **Why it works:** Since $x$ and $y$ do not appear explicitly, we seek a traveling-wave / combined-coordinate solution $z = \phi(u)$ where $\boxed{u = x + a y}$ ($a$ is an arbitrary constant).
* By the chain rule:
  $$\boxed{p = \frac{\partial z}{\partial x} = \frac{dz}{du}\frac{\partial u}{\partial x} = \frac{dz}{du}, \qquad q = \frac{\partial z}{\partial y} = \frac{dz}{du}\frac{\partial u}{\partial y} = a\frac{dz}{du}}$$
  Substituting $p = \frac{dz}{du}$ and $q = a\frac{dz}{du}$ turns the PDE $f(z, p, q) = 0$ into an **Ordinary Differential Equation (ODE)** in $z$ and $u$!
* **📖 Worked Example 1 (Notes p. 20 / PDF pp. 21–22):** Solve $9(p^2 z + q^2) = 4$.
  1. Notice $x$ and $y$ are absent! Set $u = x + a y$, so $p = \frac{dz}{du}$ and $q = a\frac{dz}{du}$.
  2. Substitute into the PDE:
     $$9\left[z\left(\frac{dz}{du}\right)^2 + a^2\left(\frac{dz}{du}\right)^2\right] = 4 \implies 9(z + a^2)\left(\frac{dz}{du}\right)^2 = 4$$
  3. Take the square root and separate variables ($z$ on the left, $u$ on the right):
     $$\frac{dz}{du} = \pm \frac{2}{3\sqrt{z + a^2}} \implies \sqrt{z + a^2}\,dz = \pm \frac{2}{3}\,du$$
  4. Integrate both sides ($\int (z+a^2)^{1/2}dz = \frac{2}{3}(z+a^2)^{3/2}$):
     $$\frac{2}{3}(z + a^2)^{3/2} = \pm \frac{2}{3}u + c' \implies (z + a^2)^{3/2} = \pm u + b = \pm(x + a y) + b$$
  5. Square both sides to get the complete integral: $\boxed{(z + a^2)^3 = (x + a y + b)^2}$.

---
#### 🔹 Form 4: Separable Equations — $f_1(x, p) = f_2(y, q)$ (Notes pp. 22–24 / PDF pp. 23–25)
* **Why it works:** When $z$ is absent and all $x, p$ terms can be moved to the left while all $y, q$ terms are moved to the right:
  $$f_1(x, p) = f_2(y, q) = a \quad (\text{arbitrary constant})$$
  Why must both sides equal a constant $a$? Because $(x, p)$ and $(y, q)$ can be varied independently! We then solve $f_1(x, p) = a$ for $p = g_1(x, a)$ and $f_2(y, q) = a$ for $q = g_2(y, a)$, and substitute into the **total differential of $z(x, y)$**:
  $$\boxed{dz = p\,dx + q\,dy \implies z = \int g_1(x, a)\,dx + \int g_2(y, a)\,dy + b}$$
* **📖 Worked Example 1 (Notes p. 23 / PDF p. 24):** Solve $p - x^2 = q + y^2$.
  1. Notice the left side depends only on $(x, p)$ and the right side depends only on $(y, q)$! Set both equal to $a$:
     $$p - x^2 = q + y^2 = a \implies p = x^2 + a, \qquad q = a - y^2$$
  2. Substitute $p$ and $q$ into $dz = p\,dx + q\,dy$:
     $$dz = (x^2 + a)\,dx + (a - y^2)\,dy$$
  3. Integrate term by term:
     $$\boxed{z = \frac{1}{3}x^3 + a x + a y - \frac{1}{3}y^3 + b = \frac{1}{3}(x^3 - y^3) + a(x + y) + b}$$

---
### Part B: Charpit's General Method & Full Walkthrough of Example 2 (Notes pp. 24–26 / PDF pp. 25–27)

What if $f(x, y, z, p, q) = 0$ contains coupled variables so none of Forms 1–4 apply?
**Charpit's Method** is the universal master algorithm! Our goal is to find a **second equation relating $p$ and $q$** (with an arbitrary constant $a$) so that we can solve for $p$ and $q$ and integrate $dz = p\,dx + q\,dy$.
To make $dz = p\,dx + q\,dy$ integrable, $(p, q)$ must satisfy **Charpit's Auxiliary Equations** (**Notes p. 25, Eq. 5 / PDF p. 26**):

$$\boxed{\frac{dp}{f_x + p f_z} = \frac{dq}{f_y + q f_z} = \frac{dz}{-p f_p - q f_q} = \frac{dx}{-f_p} = \frac{dy}{-f_q}}$$

> 💡 **See how Charpit's system automatically proves Forms 1 and 2:**
> * **Form 1 ($f(p,q)=0$):** Here $f_x = f_y = f_z = 0$, so the denominators under $dp$ and $dq$ are zero $\implies dp = 0 \implies p = a$ and $dq = 0 \implies q = b$!
> * **Form 2 (Clairaut $f = px + qy + g(p,q) - z = 0$):** Here $f_x = p,\; f_y = q,\; f_z = -1$, so $f_x + p f_z = p - p = 0$ and $f_y + q f_z = q - q = 0 \implies dp = 0, dq = 0 \implies p = a, q = b$!

---
#### 📖 Full Walkthrough: Charpit's Method on Example 2 (Notes pp. 25–26 / PDF pp. 26–27)
**Problem:** Find a complete integral of the general non-linear PDE:
$$2 z x - p x^2 - 2 q x y + p q = 0$$
*(Notice all 5 variables $x, y, z, p, q$ are coupled together—Forms 1, 2, 3, and 4 all fail here!)*

* **Step 1 (Write $f(x,y,z,p,q) = 0$ and compute all 5 partial derivatives):**
  $$f(x, y, z, p, q) = 2 z x - p x^2 - 2 q x y + p q = 0$$
  $$\begin{aligned}
  f_x &= 2z - 2px - 2qy, \qquad & f_y &= -2qx, \qquad & f_z &= 2x \\
  f_p &= -x^2 + q, \qquad & f_q &= -2xy + p
  \end{aligned}$$
  > 📌 **PDF Page 27 Typo Alert (Notes p. 26):**
  > In the printed lecture notes on page 26 (PDF page 27), there is a copy-paste typo in the intermediate list of derivatives where $f_z$ and $f_p$ were accidentally written as $-2qx$, even though **the very next line of the PDF uses the correct values $f_z = 2x$ and $f_p = -x^2 + q$**! Let's verify how the correct values simplify Charpit's denominators:
  * Denominator under $dp$: $f_x + p f_z = (2z - 2px - 2qy) + p(2x) = 2z - 2qy$
  * Denominator under $dq$: $f_y + q f_z = -2qx + q(2x) = \mathbf{0}$!

* **Step 2 (Pick the simplest fraction from Charpit's Auxiliary Equations):**
  $$\frac{dp}{2z - 2qy} = \frac{dq}{\mathbf{0}} = \frac{dz}{-p(q - x^2) - q(p - 2xy)} = \frac{dx}{x^2 - q} = \frac{dy}{2xy - p}$$
  Look at the second fraction $\frac{dq}{0}$! Since its denominator is zero, its numerator must be zero:
  $$dq = 0 \implies \boxed{q = a} \quad (\text{where } a \text{ is an arbitrary constant!})$$

* **Step 3 (Substitute $q = a$ into the original PDE to solve for $p$):**
  $$2 z x - p x^2 - 2 a x y + a p = 0 \implies p(a - x^2) = 2x(ay - z) \implies \boxed{p = \frac{2x(z - ay)}{x^2 - a}}$$

* **Step 4 (Substitute $p$ and $q$ into $dz = p\,dx + q\,dy$ and integrate!):**
  $$dz = \frac{2x(z - ay)}{x^2 - a}\,dx + a\,dy \implies \underbrace{dz - a\,dy}_{d(z - ay)} = \frac{2x(z - ay)}{x^2 - a}\,dx$$
  Notice how $dz - a\,dy = d(z - ay)$ is an exact differential! Separate $(z - ay)$ and $(x^2 - a)$:
  $$\frac{d(z - ay)}{z - ay} = \frac{2x\,dx}{x^2 - a}$$
  Integrate both sides:
  $$\ln|z - ay| = \ln|x^2 - a| + \ln|b| = \ln|b(x^2 - a)| \implies z - ay = b(x^2 - a)$$
  $$\boxed{z = a y + b(x^2 - a)}$$
  We just solved a 5-variable non-linear PDE in 4 clean steps!

---
### ⚠️ Common Mistakes to Avoid
1. **Adding an extra $+ c$ to Clairaut's solution:** In $z = ax + by + f(a,b)$, $(a, b)$ are already your two independent arbitrary constants—do not add a third constant $+ c$!
2. **Planes vs. Cones:** A single complete integral of $p^2 + q^2 = 1$ with $c=0$ is a tilted plane $z = x\cos\alpha + y\sin\alpha$; the circular distance cone $x^2 + y^2 = z^2$ is the **envelope** formed by sweeping the plane direction angle $\alpha$ through all $360^\circ$.


---
## 3. 📝 Paper & Pen Challenge 1 (Notes p. 19, Example 1 / PDF p. 20)

**Problem:** Find the complete integral of Clairaut's PDE:
$$z = p x + q y + p q$$

<details>
<summary>💡 <b>Hint (Clairaut's Rule)</b></summary>

Since the PDE is in Clairaut's form $z = px + qy + f(p,q)$ with $f(p,q) = pq$, substitute $p = a$ and $q = b$ directly without adding any extra constant.
</details>


In [ ]:
# GRADED CELL: Paper Challenge 1
### START CODE HERE ### (≈ 1 line of code)
# Enter the complete integral in terms of x, y, a, b:
student_clairaut = None
### END CODE HERE ###

# Run numerical verification test:
test_lab3.check_paper_challenge_1(student_clairaut)


---
## 4. 🚀 Rover Bridge Challenge (From Plane Wavefronts to the Euclidean Distance Cone)

Let's solve the Eikonal equation governing Euclidean obstacle dilation:
$$p^2 + q^2 = 1$$

**Work through the envelope derivation on paper:**
1. By **Form 1 ($f(p, q) = 0$)**, set $p = \cos\alpha$, so $q = \sin\alpha$ (since $\cos^2\alpha + \sin^2\alpha = 1$).
2. The family of planes passing through the origin $(0,0,0)$ is:
   $$z = x\cos\alpha + y\sin\alpha$$
3. To find the **envelope** over all propagation angles $\alpha \in [0, 2\pi)$, differentiate w.r.t. $\alpha$ and set to zero:
   $$\frac{\partial z}{\partial \alpha} = -x\sin\alpha + y\cos\alpha = 0$$
4. Square both equations and add them to eliminate $\alpha$:
   $$z^2 + 0^2 = (x\cos\alpha + y\sin\alpha)^2 + (-x\sin\alpha + y\cos\alpha)^2 = x^2(\cos^2\alpha + \sin^2\alpha) + y^2(\sin^2\alpha + \cos^2\alpha)$$

Simplify the right-hand side and write the resulting distance cone equation relating $x, y, z$:

<details>
<summary>💡 <b>Hint (Pythagorean Identity)</b></summary>

Using $\cos^2\alpha + \sin^2\alpha = 1$, the right-hand side simplifies to $x^2 + y^2$, giving the circular cone equation relating $x^2 + y^2$ and $z^2$.
</details>


In [ ]:
# GRADED CELL: Rover Wavefront Challenge
### START CODE HERE ### (≈ 1 line of code)
# Write the distance cone equation relating x, y, and z:
student_envelope = None
### END CODE HERE ###

# Run numerical verification test:
test_lab3.check_paper_challenge_2(student_envelope)


---
## 5. 💻 Python Implementation: Euclidean Wavefront & Obstacle Dilation

From $x^2 + y^2 = z^2$ (shifted to a source at $(y_0, x_0)$), the positive distance field is:
$$T(x, y) = \sqrt{(x - x_0)^2 + (y - y_0)^2}$$

In Python, `np.hypot(X - source[1], Y - source[0])` evaluates this exact Euclidean distance, which is the foundation for expanding obstacles by `rover_radius` in [`terrain.dilate()`](rover_path_finder/terrain.py). Complete `compute_eikonal_wavefront` below.


In [ ]:
# GRADED FUNCTION: compute_eikonal_wavefront
def compute_eikonal_wavefront(grid_size=(50, 50), source=(25, 25)):
    # Computes the Euclidean distance field T(x, y) satisfying p^2 + q^2 = 1
    # Note: source = (row_y, col_x)
    Ny, Nx = grid_size
    Y, X = np.ogrid[:Ny, :Nx]

    ### START CODE HERE ### (≈ 1 line of code)
    # Compute Euclidean distance from (source[1], source[0]) using np.hypot:
    dist = None
    ### END CODE HERE ###

    return dist

# Run automated unit test:
test_lab3.test_eikonal_wavefront_distance(compute_eikonal_wavefront)


---
## 6. 🎨 Visualizing the Physics & Connecting to `rover_path_finder/terrain.py`

Run the cell below to see:
1. **Left:** The circular Eikonal distance field $T(x, y)$ satisfying $\|\nabla T\|^2 = p^2 + q^2 = 1$.
2. **Right:** Calling [`terrain.dilate(raw, rover_radius=3)`](rover_path_finder/terrain.py) from our rover engine to add an orange Euclidean safety buffer around every red Martian hazard!


In [ ]:
from rover_path_finder import terrain

wavefront = compute_eikonal_wavefront((60, 60), source=(30, 30))
t_data = terrain.make_terrain(N=100, seed=42, rover_radius=3)
elev, raw_hazard, buffered_hazard = t_data["h"], t_data["raw"], t_data["hazard"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

cs = axes[0].contourf(wavefront, levels=20, cmap='plasma')
plt.colorbar(cs, ax=axes[0], label="Euclidean Distance T(x, y)")
axes[0].plot(30, 30, 'wo', markersize=9, label="Source (x0, y0)")
axes[0].set_title(r"1. Eikonal Distance Field: $\|\nabla T\|^2 = p^2 + q^2 = 1$", fontsize=12)
axes[0].legend()

safety_margin_only = buffered_hazard & (~raw_hazard)
overlay = np.zeros((100, 100, 3))
overlay[~buffered_hazard] = [0.95, 0.95, 0.95]   # Safe navigable ground
overlay[safety_margin_only] = [1.0, 0.65, 0.1]   # Euclidean dilation buffer (r <= 3)
overlay[raw_hazard] = [0.75, 0.1, 0.1]           # Raw rock/crater hazards

axes[1].imshow(overlay, origin='upper')
axes[1].set_title("2. Real Engine: terrain.dilate(raw, rover_radius=3)", fontsize=12)
axes[1].set_xlabel("x (pixels)")
axes[1].set_ylabel("y (pixels)")

plt.tight_layout()
plt.show()

print("Lab 3 Takeaways:")
print("  1) Non-linear 1st-order PDEs are solved via Standard Forms 1-4 or Charpit's Auxiliary Equations.")
print("  2) The Eikonal equation p^2 + q^2 = 1 generates the circular distance cone x^2 + y^2 = z^2.")
print("  3) terrain.dilate() uses this Euclidean distance to inflate hazards by the rover's physical radius.")
